# Coding CC1 manual baseline

In [76]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

from evals_local.metrics import label_metrics
from evals_local.normalizers import to_label_set, normalize_threat_labels, normalize_region_labels, normalize_geo_labels

In [77]:
annotated_path = repo_root / "data/consistency-check-datasets/data-coding/annotated_coding_dataset.xlsx"
labels_root = repo_root / "data/labels"

sheet_to_truth = {
    "l1_manual_sample": labels_root / "l1" / "L1) driver set.csv",
    "l2_manual_sample": labels_root / "l2" / "L2) threats set.csv",
    "l3_manual_sample": labels_root / "l3" / "L3) geography set.csv",
    "l4_manual_sample": labels_root / "l4" / "L4) ecosystem set.csv",
    "l5_manual_sample": labels_root / "l5" / "L5) study set.csv",
    "l6_manual_sample": labels_root / "l6" / "L6) taxa set.csv",
}

In [78]:
def merge_manual_truth(sheet_name):
    manual = pd.read_excel(annotated_path, sheet_name=sheet_name).copy()
    truth = pd.read_csv(sheet_to_truth[sheet_name]).copy()
    manual["UT (Unique WOS ID)"] = manual["UT (Unique WOS ID)"].astype(str).str.strip()
    truth["UT (Unique WOS ID)"] = truth["UT (Unique WOS ID)"].astype(str).str.strip()
    return manual.merge(
        truth,
        on="UT (Unique WOS ID)",
        how="left",
        validate="one_to_one",
        suffixes=("_pred", ""),
    )


def weighted_row(df, truth_col, pred_col, normalizer, include_mask=None):
    metrics = label_metrics(df, truth_col, pred_col, normalizer=normalizer, include_mask=include_mask)
    return metrics[metrics["label"] == "_weighted"].iloc[0]


def no_banned_mask(df, cols, banned, lowercase=False):
    if lowercase:
        parts = [df[c].apply(lambda v: len({s.lower() for s in to_label_set(v)} & banned) == 0) for c in cols]
    else:
        parts = [df[c].apply(lambda v: len(to_label_set(v) & banned) == 0) for c in cols]
    return pd.concat(parts, axis=1).all(axis=1)

In [79]:
results = []
per_label_results = []
n_breakdown_results = []
full_metrics = {}

_SUMMARY_LABELS = {"_weighted", "_macro", "_micro"}


def collect(task, label_set, df, truth_col, pred_col, normalizer, include_mask=None):
    """Run label_metrics once; keep the weighted summary row, the raw per-label rows,
    and the n breakdown (source -> excluded -> scored)."""
    metrics = label_metrics(df, truth_col, pred_col, normalizer=normalizer, include_mask=include_mask)
    full_metrics[(task, label_set)] = metrics

    w = metrics[metrics["label"] == "_weighted"].iloc[0]
    results.append([task, label_set, w["precision"], w["recall"], w["f1"], w["kappa"], w["support"], w["n"]])

    per = metrics[~metrics["label"].isin(_SUMMARY_LABELS)]
    support_sum = per["support"].sum() or 1
    for _, r in per.iterrows():
        per_label_results.append([
            task, label_set, r["label"],
            int(r["support"]), r["support"] / support_sum,
            r["precision"], r["recall"], r["f1"], r["kappa"],
            int(r["tp"]), int(r["fp"]), int(r["fn"]), int(r["n"]),
        ])

    # n breakdown: how the source rows split into excluded vs scored (same normalizer/mask as metrics)
    nonempty = df[truth_col].apply(normalizer).apply(len) > 0
    if include_mask is not None:
        m = include_mask.loc[nonempty.index]
        scored = nonempty & m
        masked = nonempty & ~m
    else:
        scored = nonempty
        masked = pd.Series(False, index=nonempty.index)
    n_breakdown_results.append([
        task, label_set,
        len(df),                       # n_source
        int((~nonempty).sum()),        # n_empty_truth
        int(masked.sum()),             # n_masked (banned/special labels in truth or pred)
        int(scored.sum()),             # n_scored  (== metrics n)
    ])


# L1
df = merge_manual_truth("l1_manual_sample")
collect("L1", "driver", df, "driver", "driver_pred", to_label_set)

# L2
df = merge_manual_truth("l2_manual_sample")
_BANNED_L2 = {"no threat_l2 candidates found", "no threat_l1 candidates found", "unclear"}

mask_l0 = no_banned_mask(df, ["threats_l0", "threats_l0_pred"], _BANNED_L2)
collect("L2", "threats_l0", df, "threats_l0", "threats_l0_pred", normalize_threat_labels, include_mask=mask_l0)

# threats_l1 is only partially annotated in the manual workbook (~50/100 rows); score only
# rows that have BOTH a manual annotation and a real truth label (the notna() clause; the
# collect() scorer already drops rows whose truth is empty)
mask_l1 = no_banned_mask(df, ["threats_l1", "threats_l1_pred"], _BANNED_L2) & df["threats_l1_pred"].notna()
collect("L2", "threats_l1", df, "threats_l1", "threats_l1_pred", normalize_threat_labels, include_mask=mask_l1)

# L3
df = merge_manual_truth("l3_manual_sample")
for truth_col, pred_col, normalizer in [
    ("region", "region_pred", normalize_region_labels),
    ("sub-region", "sub-region_pred", to_label_set),
    ("country", "country_pred", normalize_geo_labels),
]:
    collect("L3", truth_col, df, truth_col, pred_col, normalizer)

# L4
df = merge_manual_truth("l4_manual_sample")
collect("L4", "realm", df, "realm", "realm_pred", to_label_set)
mask = no_banned_mask(df, ["biome", "biome_pred"], {"no biome candidates found"}, lowercase=True)
collect("L4", "biome", df, "biome", "biome_pred", to_label_set, include_mask=mask)

# L5
df = merge_manual_truth("l5_manual_sample")
collect("L5", "study_design", df, "study_design", "study_design_pred", to_label_set)

# L6
df = merge_manual_truth("l6_manual_sample")
for truth_col, pred_col in [
    ("kingdom", "kingdom_pred"),
    ("phylum", "phylum_pred"),
    ("class", "class_pred"),
    ("order", "order_pred"),
    ("genus", "genus_pred"),
    ("specie", "specie_pred"),
]:
    mask = no_banned_mask(df, [truth_col, pred_col], {"not applicable", "unclear"})
    collect("L6", truth_col, df, truth_col, pred_col, to_label_set, include_mask=mask)

summary = pd.DataFrame(results, columns=["task", "label_set", "precision", "recall", "f1", "kappa", "support", "n"])
summary

,task,label_set,precision,recall,f1,kappa,support,n
0,L1,driver,0.865955,0.794393,0.819442,0.760524,107,100
1,L2,threats_l0,0.822205,0.701493,0.750562,0.676622,134,86
2,L2,threats_l1,0.725140,0.514706,0.576220,0.530839,68,49
3,L3,region,0.950163,0.901961,0.923781,0.895433,102,100
4,L3,sub-region,0.963530,0.913978,0.933274,0.908148,93,91
5,L3,country,0.950988,0.819149,0.873886,0.846118,94,90
6,L4,realm,1.000000,0.930693,0.961446,0.936470,101,100
7,L4,biome,1.000000,1.000000,1.000000,NaN,1,1
8,L5,study_design,0.823071,0.764706,0.767632,0.526580,102,100
9,L6,kingdom,0.991250,1.000000,0.995333,0.994957,100,100


In [80]:
summary.assign(
    precision=(summary["precision"] * 100).round(2),
    recall=(summary["recall"] * 100).round(2),
    f1=(summary["f1"] * 100).round(2),
    kappa=(summary["kappa"] * 100).round(2),
)

,task,label_set,precision,recall,f1,kappa,support,n
0,L1,driver,86.60,79.44,81.94,76.05,107,100
1,L2,threats_l0,82.22,70.15,75.06,67.66,134,86
2,L2,threats_l1,72.51,51.47,57.62,53.08,68,49
3,L3,region,95.02,90.20,92.38,89.54,102,100
4,L3,sub-region,96.35,91.40,93.33,90.81,93,91
5,L3,country,95.10,81.91,87.39,84.61,94,90
6,L4,realm,100.00,93.07,96.14,93.65,101,100
7,L4,biome,100.00,100.00,100.00,NaN,1,1
8,L5,study_design,82.31,76.47,76.76,52.66,102,100
9,L6,kingdom,99.12,100.00,99.53,99.50,100,100


## Raw per-label breakdown

The `summary` above is the support-weighted average over each label set. Below is the raw
per-label detail that makes up those weights: within a label set,
`weight = support / Σ support`, and the weighted metric equals `Σ (weight × metric)`.

In [81]:
per_label = pd.DataFrame(
    per_label_results,
    columns=["task", "label_set", "label", "support", "weight",
             "precision", "recall", "f1", "kappa", "tp", "fp", "fn", "n"],
).sort_values(["task", "label_set", "support"], ascending=[True, True, False]).reset_index(drop=True)
per_label

,task,label_set,label,support,weight,precision,recall,f1,kappa,tp,fp,fn,n
0,L1,driver,pollution,39,0.364486,0.914286,0.820513,0.864865,0.785867,32,3,7,100
1,L1,driver,direct exploitation and resource extraction,24,0.224299,0.937500,0.625000,0.750000,0.690594,15,1,9,100
2,L1,driver,land/sea use change,22,0.205607,0.750000,0.954545,0.840000,0.787686,21,7,1,100
3,L1,driver,climate change,13,0.121495,0.692308,0.692308,0.692308,0.646331,9,4,4,100
4,L1,driver,invasive alien species,9,0.084112,1.000000,0.888889,0.941176,0.935733,8,0,1,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,L6,specie,sparus aurata,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,0,85
241,L6,specie,toxoplasma gondii,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,0,85
242,L6,specie,ursus arctos,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,0,85
243,L6,specie,vulpes lagopus,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,0,85


In [82]:
# Readable view: metrics and weight as percentages (mirrors the summary display above)
per_label.assign(
    weight=(per_label["weight"] * 100).round(2),
    precision=(per_label["precision"] * 100).round(2),
    recall=(per_label["recall"] * 100).round(2),
    f1=(per_label["f1"] * 100).round(2),
    kappa=(per_label["kappa"] * 100).round(2),
)

,task,label_set,label,support,weight,precision,recall,f1,kappa,tp,fp,fn,n
0,L1,driver,pollution,39,36.45,91.43,82.05,86.49,78.59,32,3,7,100
1,L1,driver,direct exploitation and resource extraction,24,22.43,93.75,62.50,75.00,69.06,15,1,9,100
2,L1,driver,land/sea use change,22,20.56,75.00,95.45,84.00,78.77,21,7,1,100
3,L1,driver,climate change,13,12.15,69.23,69.23,69.23,64.63,9,4,4,100
4,L1,driver,invasive alien species,9,8.41,100.00,88.89,94.12,93.57,8,0,1,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,L6,specie,sparus aurata,0,0.00,0.00,0.00,0.00,0.00,0,1,0,85
241,L6,specie,toxoplasma gondii,0,0.00,0.00,0.00,0.00,0.00,0,1,0,85
242,L6,specie,ursus arctos,0,0.00,0.00,0.00,0.00,0.00,0,1,0,85
243,L6,specie,vulpes lagopus,0,0.00,0.00,0.00,0.00,0.00,0,1,0,85


In [83]:
# Sanity check: weighted metrics reconstructed from the raw per-label rows match `summary`
recon = (
    per_label.assign(
        precision=per_label["precision"] * per_label["weight"],
        recall=per_label["recall"] * per_label["weight"],
        f1=per_label["f1"] * per_label["weight"],
    )
    .groupby(["task", "label_set"], as_index=False)[["precision", "recall", "f1"]]
    .sum()
)
check = summary.merge(recon, on=["task", "label_set"], suffixes=("_summary", "_recon"))
for m in ["precision", "recall", "f1"]:
    assert (check[f"{m}_summary"] - check[f"{m}_recon"]).abs().max() < 1e-9, m
print("OK: per-label weights reproduce the weighted summary metrics")

OK: per-label weights reproduce the weighted summary metrics


## n breakdown (per label set)

How the source documents split before scoring. `n` is a per-label-set quantity (identical for
every label within a set), so the breakdown is at `task + label_set` granularity:
`n_source = n_empty_truth + n_masked + n_scored`, where `n_scored` is the `n` used everywhere else.

In [84]:
n_breakdown = pd.DataFrame(
    n_breakdown_results,
    columns=["task", "label_set", "n_source", "n_empty_truth", "n_masked", "n_scored"],
)
# guard: the parts must add up, and n_scored must match the metrics n
assert (n_breakdown["n_empty_truth"] + n_breakdown["n_masked"] + n_breakdown["n_scored"]
        == n_breakdown["n_source"]).all()
assert n_breakdown.set_index(["task", "label_set"])["n_scored"].equals(
    summary.set_index(["task", "label_set"])["n"].astype(int)
)
n_breakdown

,task,label_set,n_source,n_empty_truth,n_masked,n_scored
0,L1,driver,100,0,0,100
1,L2,threats_l0,100,0,14,86
2,L2,threats_l1,100,4,47,49
3,L3,region,100,0,0,100
4,L3,sub-region,100,9,0,91
5,L3,country,100,10,0,90
6,L4,realm,100,0,0,100
7,L4,biome,100,99,0,1
8,L5,study_design,100,0,0,100
9,L6,kingdom,100,0,0,100


## Macro / weighted / micro summary

All three averages per label set, computed from `full_metrics`. `n` = documents scored;
`n_decisions` = rows × labels (the flattened grid that micro pools over).

In [85]:
# All three averages (macro / weighted / micro) per label set, from full_metrics
_avg_rows = {"_macro": "macro", "_weighted": "weighted", "_micro": "micro"}
_records = []
for (task, label_set), m in full_metrics.items():
    scored_rows = int(m[m["label"] == "_macro"].iloc[0]["n"])   # documents scored
    n_decisions = int(m[m["label"] == "_micro"].iloc[0]["n"])   # rows x labels (flattened grid)
    for key, name in _avg_rows.items():
        r = m[m["label"] == key].iloc[0]
        _records.append([task, label_set, name, r["precision"], r["recall"], r["f1"], r["kappa"],
                         r["support"], scored_rows, n_decisions])

summary_all = pd.DataFrame(
    _records,
    columns=["task", "label_set", "average", "precision", "recall", "f1", "kappa",
             "support", "n", "n_decisions"],
)
summary_all["average"] = pd.Categorical(summary_all["average"], ["macro", "weighted", "micro"], ordered=True)
summary_all = summary_all.sort_values(["task", "label_set", "average"]).reset_index(drop=True)
summary_all

,task,label_set,average,precision,recall,f1,kappa,support,n,n_decisions
0,L1,driver,macro,0.613442,0.568751,0.584050,0.549459,107,100,700
1,L1,driver,weighted,0.865955,0.794393,0.819442,0.760524,107,100,700
2,L1,driver,micro,0.765766,0.794393,0.779817,0.739224,107,100,700
3,L2,threats_l0,macro,0.771411,0.614716,0.667943,0.622774,134,86,860
4,L2,threats_l0,weighted,0.822205,0.701493,0.750562,0.676622,134,86,860
5,L2,threats_l0,micro,0.817391,0.701493,0.755020,0.713834,134,86,860
6,L2,threats_l1,macro,0.369151,0.264089,0.285477,0.269594,68,49,1127
7,L2,threats_l1,weighted,0.725140,0.514706,0.576220,0.530839,68,49,1127
8,L2,threats_l1,micro,0.583333,0.514706,0.546875,0.519707,68,49,1127
9,L3,country,macro,0.763228,0.706361,0.726903,0.722718,94,90,2430


## Combined per-label workbook

Saves one `.xlsx` to `data/labels/eval/coding_cc1_baseline/` containing: a **README** sheet,
a **summary** sheet (the macro/weighted/micro table above), and one **detail sheet per
L-level + label set** (`<task>_<label_set>`, e.g. `L3_region`). Each detail sheet lists the
raw per-label rows with the `_macro` / `_weighted` / `_micro` aggregates appended as the last
three rows — so the standalone macro/micro file is no longer needed.

In [86]:
# Build the per-label sheets (raw rows + macro/weighted/micro footer) and one workbook
per_label_summary = per_label[
    ["task", "label_set", "label", "support", "weight",
     "precision", "recall", "f1", "kappa", "tp", "fp", "fn", "n"]
].copy()

artifacts_dir = repo_root / "data/labels/eval/coding_cc1_baseline"
artifacts_dir.mkdir(parents=True, exist_ok=True)
per_label_summary_path = artifacts_dir / "per_label_summary.xlsx"

_detail_cols = ["label", "support", "weight", "precision", "recall", "f1", "kappa", "tp", "fp", "fn", "n"]


def _sheet_name(task, label_set):
    name = f"{task}_{label_set}"
    for ch in r"\/*?:[]":
        name = name.replace(ch, "-")
    return name[:31]  # Excel sheet-name limit


def _footer(task, label_set):
    """macro/weighted/micro rows for one label set, aligned to the detail columns."""
    sub = summary_all[(summary_all["task"] == task) & (summary_all["label_set"] == label_set)]
    rows = [{
        "label": f"_{r['average']}", "support": r["support"], "weight": pd.NA,
        "precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "kappa": r["kappa"],
        "tp": pd.NA, "fp": pd.NA, "fn": pd.NA, "n": r["n"],
    } for _, r in sub.iterrows()]
    return pd.DataFrame(rows, columns=_detail_cols)


per_label_readme = pd.DataFrame(
    [
        ("(sheets)", "README (this), summary (macro/weighted/micro for every label set), n_breakdown (how source docs split into excluded vs scored), then one detail sheet per L-level + label set named <task>_<label_set> (e.g. L3_region)."),
        ("(footer rows)", "The last 3 rows of each detail sheet -- _macro, _weighted, _micro -- are that label set's aggregate averages; weight/tp/fp/fn are blank for them."),
        ("label", "Individual label value within the label set (or _macro/_weighted/_micro in footer rows)."),
        ("support", "True instances of this label across scored docs (tp + fn); for footer rows, the label-set total."),
        ("weight", "support / sum(support) in the label set; this label's weight in the weighted average."),
        ("precision", "tp / (tp + fp). Of the labels predicted, the fraction that were correct."),
        ("recall", "tp / (tp + fn). Of the true labels, the fraction that were recovered."),
        ("f1", "Harmonic mean of precision and recall: 2*P*R / (P + R)."),
        ("kappa", "Cohen's kappa: agreement corrected for chance."),
        ("tp", "True positives: docs where this label is in both truth and prediction."),
        ("fp", "False positives: docs where this label is predicted but not in truth."),
        ("fn", "False negatives: docs where this label is in truth but not predicted."),
        ("n", "Documents scored in the label set (== n_scored on the n_breakdown sheet)."),
        ("average", "summary sheet only: macro = unweighted mean over labels; weighted = support-weighted mean; micro = pooled tp/fp/fn over all label-row decisions."),
        ("n_decisions", "summary sheet only: n x number of candidate labels; the grid micro pools over."),
        ("n_source", "n_breakdown sheet: rows in the merged truth+manual source for the label set."),
        ("n_empty_truth", "n_breakdown sheet: rows excluded because the normalized truth set is empty (blank or unmatched on merge)."),
        ("n_masked", "n_breakdown sheet: rows excluded by the label set's exclusion rule (banned/special labels such as 'unclear' / 'not applicable' / 'no ... candidates found' in truth OR prediction)."),
        ("n_scored", "n_breakdown sheet: rows actually scored = n_source - n_empty_truth - n_masked (this is the metrics n)."),
    ],
    columns=["column", "description"],
)

with pd.ExcelWriter(per_label_summary_path, engine="openpyxl") as writer:
    per_label_readme.to_excel(writer, sheet_name="README", index=False)
    summary_all.to_excel(writer, sheet_name="summary", index=False)
    n_breakdown.to_excel(writer, sheet_name="n_breakdown", index=False)
    for (task, label_set), group in per_label_summary.groupby(["task", "label_set"], sort=True):
        sheet = _sheet_name(task, label_set)
        body = group[_detail_cols]
        body.to_excel(writer, sheet_name=sheet, index=False)
        # append macro/weighted/micro just below the body (no header) -> avoids all-NA concat
        _footer(task, label_set).to_excel(
            writer, sheet_name=sheet, index=False, header=False, startrow=len(body) + 1
        )

n_sheets = per_label_summary.groupby(["task", "label_set"]).ngroups
print(f"Saved per_label_summary.xlsx: README + summary + n_breakdown + {n_sheets} label-set sheets "
      f"(each with a macro/weighted/micro footer) to {per_label_summary_path}")

Saved per_label_summary.xlsx: README + summary + n_breakdown + 15 label-set sheets (each with a macro/weighted/micro footer) to /Users/hunain/d/coding-projects/biodiversity/data/labels/eval/coding_cc1_baseline/per_label_summary.xlsx


In [87]:
# Readable view of the macro/weighted/micro summary (metrics as percentages)
summary_all.assign(
    precision=(summary_all["precision"] * 100).round(2),
    recall=(summary_all["recall"] * 100).round(2),
    f1=(summary_all["f1"] * 100).round(2),
    kappa=(summary_all["kappa"] * 100).round(2),
)

,task,label_set,average,precision,recall,f1,kappa,support,n,n_decisions
0,L1,driver,macro,61.34,56.88,58.40,54.95,107,100,700
1,L1,driver,weighted,86.60,79.44,81.94,76.05,107,100,700
2,L1,driver,micro,76.58,79.44,77.98,73.92,107,100,700
3,L2,threats_l0,macro,77.14,61.47,66.79,62.28,134,86,860
4,L2,threats_l0,weighted,82.22,70.15,75.06,67.66,134,86,860
5,L2,threats_l0,micro,81.74,70.15,75.50,71.38,134,86,860
6,L2,threats_l1,macro,36.92,26.41,28.55,26.96,68,49,1127
7,L2,threats_l1,weighted,72.51,51.47,57.62,53.08,68,49,1127
8,L2,threats_l1,micro,58.33,51.47,54.69,51.97,68,49,1127
9,L3,country,macro,76.32,70.64,72.69,72.27,94,90,2430
